In [7]:
import cv2
import numpy as np
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort

# ==============================
# SETTINGS
# ==============================

VIDEO_PATH = r"C:\Users\natra\Downloads\OpelInsignia\OpelInsignia\OpelInsignia_35.MP4"

DISTANCE_METERS = 12   # adjust for real distance between zones

# Load YOLO model
model = YOLO("yolov8n.pt")

# Initialize tracker
tracker = DeepSort(max_age=30)

# ==============================
# SPEED ZONES
# ==============================

start_area = [
    (50,180),
    (1230,180),
    (1230,210),
    (50,210)
]

stop_area = [
    (50,300),
    (1230,300),
    (1230,330),
    (50,330)
]

# ==============================
# VARIABLES
# ==============================

vehicle_start_frame = {}
vehicle_speed = {}

# ==============================
# FUNCTION
# ==============================

def inside_polygon(point, polygon):
    return cv2.pointPolygonTest(
        np.array(polygon, np.int32),
        point,
        False
    ) >= 0


# ==============================
# VIDEO LOOP
# ==============================

cap = cv2.VideoCapture(VIDEO_PATH)

fps = cap.get(cv2.CAP_PROP_FPS)
frame_number = 0

while True:

    ret, frame = cap.read()
    if not ret:
        break

    frame_number += 1

    # ==============================
    # YOLO DETECTION
    # ==============================

    results = model(frame, classes=[2,3,5,7], verbose=False)

    detections = []

    if len(results[0].boxes) > 0:

        boxes = results[0].boxes.xyxy.cpu().numpy()
        scores = results[0].boxes.conf.cpu().numpy()

        for box, score in zip(boxes, scores):

            x1, y1, x2, y2 = box

            detections.append(
                ([x1, y1, x2-x1, y2-y1], score, "vehicle")
            )

    # ==============================
    # TRACK VEHICLES
    # ==============================

    tracks = tracker.update_tracks(detections, frame=frame)

    for track in tracks:

        if not track.is_confirmed():
            continue

        track_id = track.track_id

        l, t, r, b = track.to_ltrb()

        x1, y1, x2, y2 = int(l), int(t), int(r), int(b)

        cx = int((x1 + x2) / 2)
        cy = int((y1 + y2) / 2)

        center = (cx, cy)

        # ==============================
        # START ZONE
        # ==============================

        if inside_polygon(center, start_area) and track_id not in vehicle_start_frame:
            vehicle_start_frame[track_id] = frame_number

        # ==============================
        # STOP ZONE
        # ==============================

        if track_id in vehicle_start_frame and track_id not in vehicle_speed:

            if inside_polygon(center, stop_area):

                start_frame = vehicle_start_frame[track_id]

                frames_taken = frame_number - start_frame

                time_seconds = frames_taken / fps

                if time_seconds > 0:

                    speed_ms = DISTANCE_METERS / time_seconds
                    speed_kmh = speed_ms * 3.6

                    vehicle_speed[track_id] = speed_kmh

        # ==============================
        # DRAW BOX + SPEED
        # ==============================

        if track_id in vehicle_speed:

            speed = int(vehicle_speed[track_id])
            label = f"{speed} km/h"
            color = (0,0,255)

            cv2.putText(
                frame,
                label,
                (x1, y1-10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (255,255,255),
                2
            )

        cv2.rectangle(frame,(x1,y1),(x2,y2),color if track_id in vehicle_speed else (0,255,0),2)

        cv2.circle(frame, center, 4, (255,0,0), -1)

    # ==============================
    # DRAW SPEED ZONES
    # ==============================

    cv2.polylines(
        frame,
        [np.array(start_area,np.int32)],
        True,
        (0,255,0),
        2
    )

    cv2.polylines(
        frame,
        [np.array(stop_area,np.int32)],
        True,
        (0,0,255),
        2
    )

    # ==============================
    # DISPLAY
    # ==============================

    cv2.imshow("Vehicle Speed Estimation", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break


cap.release()
cv2.destroyAllWindows()

In [1]:
import cv2
import numpy as np
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort

# ==============================
# SETTINGS
# ==============================

VIDEO_PATH = r"C:\Users\natra\Downloads\videoplayback.mp4"
OUTPUT_PATH = r"C:\Users\natra\Downloads\cv model outputs\physics based model\top_veiw_output_speed.mp4"

DISTANCE_METERS = 12

# Load YOLO model
model = YOLO("yolov8n.pt")

# Initialize tracker
tracker = DeepSort(max_age=30)

# ==============================
# SPEED ZONES
# ==============================

start_area = [
    (50,180),
    (1230,180),
    (1230,210),
    (50,210)
]

stop_area = [
    (50,300),
    (1230,300),
    (1230,330),
    (50,330)
]

# ==============================
# VARIABLES
# ==============================

vehicle_start_frame = {}
vehicle_speed = {}

# ==============================
# FUNCTION
# ==============================

def inside_polygon(point, polygon):
    return cv2.pointPolygonTest(
        np.array(polygon, np.int32),
        point,
        False
    ) >= 0


# ==============================
# VIDEO LOOP
# ==============================

cap = cv2.VideoCapture(VIDEO_PATH)

# 👉 ADD VIDEO WRITER
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (width, height))

frame_number = 0

while True:

    ret, frame = cap.read()
    if not ret:
        break

    frame_number += 1

    # ==============================
    # YOLO DETECTION
    # ==============================

    results = model(frame, classes=[2,3,5,7], verbose=False)

    detections = []

    if len(results[0].boxes) > 0:

        boxes = results[0].boxes.xyxy.cpu().numpy()
        scores = results[0].boxes.conf.cpu().numpy()

        for box, score in zip(boxes, scores):

            x1, y1, x2, y2 = box

            detections.append(
                ([x1, y1, x2-x1, y2-y1], score, "vehicle")
            )

    # ==============================
    # TRACK VEHICLES
    # ==============================

    tracks = tracker.update_tracks(detections, frame=frame)

    for track in tracks:

        if not track.is_confirmed():
            continue

        track_id = track.track_id

        l, t, r, b = track.to_ltrb()

        x1, y1, x2, y2 = int(l), int(t), int(r), int(b)

        cx = int((x1 + x2) / 2)
        cy = int((y1 + y2) / 2)

        center = (cx, cy)

        # ==============================
        # START ZONE
        # ==============================

        if inside_polygon(center, start_area) and track_id not in vehicle_start_frame:
            vehicle_start_frame[track_id] = frame_number

        # ==============================
        # STOP ZONE
        # ==============================

        if track_id in vehicle_start_frame and track_id not in vehicle_speed:

            if inside_polygon(center, stop_area):

                start_frame = vehicle_start_frame[track_id]

                frames_taken = frame_number - start_frame

                time_seconds = frames_taken / fps

                if time_seconds > 0:

                    speed_ms = DISTANCE_METERS / time_seconds
                    speed_kmh = speed_ms * 3.6

                    vehicle_speed[track_id] = speed_kmh

        # ==============================
        # DRAW BOX + SPEED
        # ==============================

        if track_id in vehicle_speed:

            speed = int(vehicle_speed[track_id])
            label = f"{speed} km/h"
            color = (0,0,255)

            cv2.putText(
                frame,
                label,
                (x1, y1-10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (255,255,255),
                2
            )

        else:
            color = (0,255,0)

        cv2.rectangle(frame,(x1,y1),(x2,y2),color,2)
        cv2.circle(frame, center, 4, (255,0,0), -1)

    # ==============================
    # DRAW SPEED ZONES
    # ==============================

    cv2.polylines(
        frame,
        [np.array(start_area,np.int32)],
        True,
        (0,255,0),
        2
    )

    cv2.polylines(
        frame,
        [np.array(stop_area,np.int32)],
        True,
        (0,0,255),
        2
    )

    # 👉 SAVE FRAME
    out.write(frame)

    # ==============================
    # DISPLAY
    # ==============================

    cv2.imshow("Vehicle Speed Estimation", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break


cap.release()
out.release()   # 👉 IMPORTANT
cv2.destroyAllWindows()

print("Output video saved at:", OUTPUT_PATH)

Output video saved at: C:\Users\natra\Downloads\cv model outputs\physics based model\top_veiw_output_speed.mp4


In [17]:
import cv2
import numpy as np
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort

# ==============================
# SETTINGS
# ==============================

VIDEO_PATH = r"C:\Users\natra\Downloads\OpelInsignia\OpelInsignia\OpelInsignia_35.MP4"
OUTPUT_PATH = r"C:\Users\natra\Downloads\cv model outputs\physics based model\side_veiw_output_speed.mp4"

DISTANCE_METERS = 12

# Load YOLO model
model = YOLO("yolov8n.pt")

# Initialize tracker
tracker = DeepSort(max_age=30)

# ==============================
# CORRECT SPEED ZONES (FIXED)
# ==============================

# ✅ START LINE (on road - upper)
start_area = [
    (120,480),
    (500,480),
    (520,520),
    (100,520)
]

# ✅ END LINE (closer to camera - lower)
stop_area = [
    (150,620),
    (900,620),
    (1000,700),
    (120,700)
]

# ==============================
# VARIABLES
# ==============================

vehicle_start_frame = {}
vehicle_speed = {}

# ==============================
# FUNCTION
# ==============================

def inside_polygon(point, polygon):
    return cv2.pointPolygonTest(
        np.array(polygon, np.int32),
        point,
        False
    ) >= 0


# ==============================
# VIDEO LOOP
# ==============================

cap = cv2.VideoCapture(VIDEO_PATH)

# 👉 Video writer
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (width, height))

frame_number = 0

while True:

    ret, frame = cap.read()
    if not ret:
        break

    frame_number += 1

    # ==============================
    # YOLO DETECTION
    # ==============================

    results = model(frame, classes=[2,3,5,7], verbose=False)

    detections = []

    if len(results[0].boxes) > 0:

        boxes = results[0].boxes.xyxy.cpu().numpy()
        scores = results[0].boxes.conf.cpu().numpy()

        for box, score in zip(boxes, scores):

            x1, y1, x2, y2 = box

            detections.append(
                ([x1, y1, x2-x1, y2-y1], score, "vehicle")
            )

    # ==============================
    # TRACK VEHICLES
    # ==============================

    tracks = tracker.update_tracks(detections, frame=frame)

    for track in tracks:

        if not track.is_confirmed():
            continue

        track_id = track.track_id

        l, t, r, b = track.to_ltrb()

        x1, y1, x2, y2 = int(l), int(t), int(r), int(b)

        cx = int((x1 + x2) / 2)
        cy = int((y1 + y2) / 2)

        center = (cx, cy)

        # ==============================
        # START ZONE
        # ==============================

        if inside_polygon(center, start_area) and track_id not in vehicle_start_frame:
            vehicle_start_frame[track_id] = frame_number

        # ==============================
        # STOP ZONE
        # ==============================

        if track_id in vehicle_start_frame and track_id not in vehicle_speed:

            if inside_polygon(center, stop_area):

                start_frame = vehicle_start_frame[track_id]

                frames_taken = frame_number - start_frame

                time_seconds = frames_taken / fps

                if time_seconds > 0:

                    speed_ms = DISTANCE_METERS / time_seconds
                    speed_kmh = speed_ms * 3.6

                    vehicle_speed[track_id] = speed_kmh

        # ==============================
        # DRAW BOX + SPEED
        # ==============================

        if track_id in vehicle_speed:

            speed = int(vehicle_speed[track_id])
            label = f"{speed} km/h"
            color = (0,0,255)

            cv2.putText(
                frame,
                label,
                (x1, y1-10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (255,255,255),
                2
            )

        else:
            color = (0,255,0)

        cv2.rectangle(frame,(x1,y1),(x2,y2),color,2)
        cv2.circle(frame, center, 4, (255,0,0), -1)

    # ==============================
    # DRAW ZONES
    # ==============================

    cv2.polylines(
        frame,
        [np.array(start_area,np.int32)],
        True,
        (0,255,0),
        2
    )

    cv2.polylines(
        frame,
        [np.array(stop_area,np.int32)],
        True,
        (0,0,255),
        2
    )

    # ==============================
    # SAVE + DISPLAY
    # ==============================

    out.write(frame)

    cv2.imshow("Vehicle Speed Estimation", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break
def click_event(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN:
        print(f"({x},{y})")

cv2.namedWindow("frame")
cv2.setMouseCallback("frame", click_event)

cap.release()
out.release()
cv2.destroyAllWindows()

print("Output saved at:", OUTPUT_PATH)

Output saved at: C:\Users\natra\Downloads\cv model outputs\physics based model\side_veiw_output_speed.mp4
